# Crypto Candlestick Viewer
Reads raw **1-minute** Binance perpetual data from `data_store/crypto/raw_1m/`  
and aggregates it to any supported timeframe for local interactive charting.

**Supported intervals:** `1m` · `3m` · `5m` · `15m` · `1h` · `4h`

Set your parameters in **Cell 2** then run all cells.

In [1]:
# ── 1. Setup ──────────────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = 'notebook'

from data_fetch.crypto_fetch import (
    load_crypto_ohlcv,
    resample_crypto,
    list_crypto_stored,
    DERIVED_INTERVALS,
    CRYPTO_SYMBOLS,
    _load_raw,
)

print('✅ Imports OK')
print(f'Tracked symbols : {CRYPTO_SYMBOLS}')
print(f'Derived TFs     : {list(DERIVED_INTERVALS.keys())}')

✅ Imports OK
Tracked symbols : ['BTC', 'ETH', 'SOL', 'BNB', 'XRP', 'LINK', 'AVAX', 'DOGE', 'ADA', 'DOT']
Derived TFs     : ['3m', '5m', '15m', '1h', '4h']


In [2]:
# ── 2. Parameters ─────────────────────────────────────────────────────────────
BASE      = 'BTC'      # Symbol base: BTC | ETH | SOL | BNB | XRP | LINK | AVAX | DOGE | ADA | DOT
INTERVAL  = '15m'      # Target timeframe: 1m | 3m | 5m | 15m | 1h | 4h
LOOKBACK  = 60         # Days of data to show in the chart (None = all available)
SHOW_VOL  = True       # Show volume subplot below candles
SHOW_VWAP = True       # Overlay rolling VWAP on the chart

In [3]:
# ── 3. Available data inventory ───────────────────────────────────────────────
stored = list_crypto_stored()
if not stored:
    print('⚠  No crypto data found in data_store/crypto/raw_1m/')
    print('   Run the Data Fetch page in the web UI, or call fetch_crypto_1m() to download data.')
else:
    inv = pd.DataFrame(stored)[['symbol', 'rows', 'first_date', 'last_date', 'size_kb']]
    inv['rows'] = inv['rows'].apply(lambda x: f'{x:,}')
    inv['size_kb'] = inv['size_kb'].apply(lambda x: f'{x:,.0f} KB')
    display(inv)

,symbol,rows,first_date,last_date,size_kb
0,ADAUSDT_PERP,"649,500",2024-03-31,2025-06-25,"26,510 KB"
1,AVAXUSDT_PERP,"1,051,199",2024-03-31,2026-03-31,"35,195 KB"
2,BNBUSDT_PERP,"504,000",2024-03-31,2025-03-16,"19,947 KB"
3,BTCUSDT_PERP,"1,051,199",2024-03-31,2026-03-31,"53,183 KB"
4,DOGEUSDT_PERP,"466,500",2024-03-31,2025-02-18,"21,429 KB"
5,DOTUSDT_PERP,"78,000",2024-03-31,2024-05-24,"3,659 KB"
6,ETHUSDT_PERP,"1,051,199",2024-03-31,2026-03-31,"53,805 KB"
7,LINKUSDT_PERP,"196,500",2024-03-31,2024-08-14,"9,350 KB"
8,SOLUSDT_PERP,"1,051,199",2024-03-31,2026-03-31,"42,271 KB"
9,XRPUSDT_PERP,"289,500",2024-03-31,2024-10-18,"12,864 KB"


In [4]:
# ── 4. Load and aggregate data ────────────────────────────────────────────────
df = load_crypto_ohlcv(BASE, INTERVAL)

if df is None or df.empty:
    raise RuntimeError(
        f'No data found for {BASE}. '
        'Fetch it via the web UI or fetch_crypto_1m() first.'
    )

# Optional date filter
if LOOKBACK:
    cutoff = df.index[-1] - pd.Timedelta(days=LOOKBACK)
    df = df[df.index >= cutoff]

symbol_label = f'{BASE}USDT PERP'
print(f'{symbol_label} | {INTERVAL} | {len(df):,} bars')
print(f'Range  : {df.index[0]}  →  {df.index[-1]}')
print(f'Columns: {list(df.columns)}')
df.tail(5)

BTCUSDT PERP | 15m | 5,761 bars
Range  : 2026-01-30 00:45:00  →  2026-03-31 00:45:00
Columns: ['open', 'high', 'low', 'close', 'volume', 'quote_volume', 'trades', 'taker_buy_base', 'taker_buy_quote']


,open,high,low,close,volume,quote_volume,trades,taker_buy_base,taker_buy_quote
open_time,,,,,,,,,
2026-03-30 23:45:00,66729.6,66769.1,66675.4,66764.4,601.844,4.015226e+07,15856,256.543,1.711685e+07
2026-03-31 00:00:00,66764.4,66792.4,66525.6,66614.5,1371.826,9.147738e+07,43454,518.315,3.456005e+07
2026-03-31 00:15:00,66615.0,66790.1,66498.9,66779.9,1232.575,8.213529e+07,43958,668.787,4.457435e+07
2026-03-31 00:30:00,66780.0,66922.8,66731.5,66912.6,1306.619,8.731673e+07,48388,785.182,5.247272e+07
2026-03-31 00:45:00,66912.6,67276.0,66906.8,67172.5,4259.105,2.857879e+08,101196,2574.030,1.727220e+08


In [5]:
# ── 5. OHLCV statistics ───────────────────────────────────────────────────────
stats = pd.DataFrame({
    'open':  df['open'],
    'high':  df['high'],
    'low':   df['low'],
    'close': df['close'],
    'volume': df['volume'],
}).describe().round(4)

# Bar range stats
df_stats = df.copy()
df_stats['bar_range']  = df_stats['high'] - df_stats['low']
df_stats['bar_range%'] = (df_stats['bar_range'] / df_stats['close'] * 100).round(4)
df_stats['bullish']    = (df_stats['close'] >= df_stats['open']).astype(int)

print(f'Bull candles : {df_stats["bullish"].sum():,} / {len(df_stats):,}  '
      f'({df_stats["bullish"].mean()*100:.1f}%)')
print(f'Avg bar range: {df_stats["bar_range%"].mean():.3f}%  '
      f'(max {df_stats["bar_range%"].max():.3f}%)')
print()
display(stats)

Bull candles : 2,895 / 5,761  (50.3%)
Avg bar range: 0.448%  (max 5.031%)



,open,high,low,close,volume
count,5761.0000,5761.0000,5761.0000,5761.0000,5761.0000
mean,69789.9953,69945.4091,69632.2286,69787.0062,2249.0466
std,3845.2464,3855.8429,3829.5507,3840.6079,2629.1655
min,60198.7000,61758.7000,59800.0000,60199.1000,137.0220
25%,67313.1000,67446.2000,67180.5000,67311.6000,838.8890
50%,68964.3000,69119.6000,68838.2000,68963.2000,1389.4510
75%,70911.8000,71053.2000,70782.0000,70911.4000,2577.6300
max,84359.2000,84599.0000,84158.5000,84264.7000,42233.4030


In [8]:
df

,open,high,low,close,volume,quote_volume,trades,taker_buy_base,taker_buy_quote
open_time,,,,,,,,,
2026-01-30 00:45:00,84359.2,84490.3,84100.0,84157.4,2066.661,1.741558e+08,41990,885.457,7.462257e+07
2026-01-30 01:00:00,84157.4,84157.5,83950.0,84104.0,1710.211,1.437694e+08,45512,809.680,6.806908e+07
2026-01-30 01:15:00,84104.1,84156.9,83468.2,83733.4,4915.308,4.115860e+08,82231,1924.170,1.611535e+08
2026-01-30 01:30:00,83733.3,83855.2,81000.0,81988.5,24754.428,2.038700e+09,383286,9235.255,7.611902e+08
2026-01-30 01:45:00,81988.5,82524.4,81558.2,82434.7,9013.483,7.402429e+08,237189,4606.194,3.783259e+08
...,...,...,...,...,...,...,...,...,...
2026-03-30 23:45:00,66729.6,66769.1,66675.4,66764.4,601.844,4.015226e+07,15856,256.543,1.711685e+07
2026-03-31 00:00:00,66764.4,66792.4,66525.6,66614.5,1371.826,9.147738e+07,43454,518.315,3.456005e+07
2026-03-31 00:15:00,66615.0,66790.1,66498.9,66779.9,1232.575,8.213529e+07,43958,668.787,4.457435e+07


In [ ]:
# ── 6. Candlestick chart ──────────────────────────────────────────────────────
def compute_vwap(df: pd.DataFrame, window: int = 50) -> pd.Series:
    """Rolling VWAP using typical price × volume."""
    typical = (df['high'] + df['low'] + df['close']) / 3
    vol     = df['volume'].replace(0, np.nan)
    return (typical * vol).rolling(window).sum() / vol.rolling(window).sum()


def plot_candles(
    df: pd.DataFrame,
    title: str,
    show_volume: bool = True,
    show_vwap: bool = True,
    vwap_window: int = 50,
) -> go.Figure:
    row_heights = [0.75, 0.25] if show_volume else [1.0]
    rows        = 2 if show_volume else 1
    print("rows: ",rows)

    fig = make_subplots(
        rows=rows, cols=1,
        shared_xaxes=True,
        row_heights=row_heights,
        vertical_spacing=0.02,
    )

    # ── Candlestick ──
    fig.add_trace(go.Candlestick(
        x=df.index,
        open=df['open'], high=df['high'],
        low=df['low'],   close=df['close'],
        name='Price',
        increasing_line_color='#22c55e',
        decreasing_line_color='#ef4444',
    ), row=1, col=1)

    # ── Rolling VWAP ──
    if show_vwap and 'volume' in df.columns:
        vwap = compute_vwap(df, window=vwap_window)
        fig.add_trace(go.Scatter(
            x=df.index, y=vwap,
            mode='lines',
            line=dict(color='#facc15', width=1.2, dash='dot'),
            name=f'VWAP({vwap_window})',
        ), row=1, col=1)

    # ── Volume ──
    if show_volume and 'volume' in df.columns:
        bull_mask = df['close'] >= df['open']

        colors = np.where(
            bull_mask,
            'rgba(38,166,154,1)',
            'rgba(239,83,80,1)'
        )

        fig.add_trace(go.Bar(
            x=df.index,
            y=df['volume'],
            marker=dict(color=colors, line=dict(width=0)),
            name='Volume',
            width=0.9,
        ), row=2, col=1)

    fig.update_layout(
        title=title,
        template='plotly_dark',
        xaxis_rangeslider_visible=False,
        height=700,
        legend=dict(x=0.01, y=0.99),
        margin=dict(l=60, r=20, t=60, b=40),
    )
    if show_volume:
        print("yes")
        fig.update_yaxes(title_text='Price',  row=1, col=1)
        fig.update_yaxes(title_text='Volume', row=2, col=1, showticklabels=False)

    return fig


title = f'{symbol_label} · {INTERVAL}  ({df.index[0].date()} → {df.index[-1].date()})'
fig   = plot_candles(df, title, show_volume=SHOW_VOL, show_vwap=SHOW_VWAP)
fig.show()

rows:  2
yes


In [ ]:
# ── 7. Multi-timeframe comparison ─────────────────────────────────────────────
# Loads all supported timeframes and plots each on its own chart.
# Useful for quick top-down structure review.

ALL_TFS    = ['1m', '3m', '5m', '15m', '1h', '4h']
LOOKBACK_MTF = 30    # days to show per TF chart

for tf in ALL_TFS:
    try:
        df_tf = load_crypto_ohlcv(BASE, tf)
        if df_tf is None or df_tf.empty:
            print(f'  ⚠  {tf}: no data')
            continue

        if LOOKBACK_MTF:
            cutoff = df_tf.index[-1] - pd.Timedelta(days=LOOKBACK_MTF)
            df_tf  = df_tf[df_tf.index >= cutoff]

        if df_tf.empty:
            print(f'  ⚠  {tf}: no data in range')
            continue

        t = f'{symbol_label} · {tf}  ({df_tf.index[0].date()} → {df_tf.index[-1].date()})'
        plot_candles(df_tf, t, show_volume=True, show_vwap=True).show()

    except Exception as e:
        print(f'  ✗  {tf}: {e}')

In [ ]:
# ── 8. Custom aggregation from raw 1m ─────────────────────────────────────────
# Build any arbitrary timeframe directly from 1m using pandas resample.
# The project's resample_crypto() only supports 3m / 5m / 15m / 1h / 4h;
# this cell lets you go beyond that (e.g. 2h, 6h, 8h, 1d).

CUSTOM_TF    = '4h'       # any valid pandas offset: '2h', '6h', '8h', '1d', '2d' …
CUSTOM_DAYS  = 90         # lookback days for this chart

_RESAMPLE_AGG = {
    'open':            'first',
    'high':            'max',
    'low':             'min',
    'close':           'last',
    'volume':          'sum',
    'quote_volume':    'sum',
    'trades':          'sum',
    'taker_buy_base':  'sum',
    'taker_buy_quote': 'sum',
}

df_1m = _load_raw(BASE)
if df_1m is None or df_1m.empty:
    print(f'⚠  No 1m raw data for {BASE}')
else:
    cutoff_custom = df_1m.index[-1] - pd.Timedelta(days=CUSTOM_DAYS)
    df_1m_trim    = df_1m[df_1m.index >= cutoff_custom]

    agg = {col: rule for col, rule in _RESAMPLE_AGG.items() if col in df_1m_trim.columns}
    df_custom = (
        df_1m_trim
        .resample(CUSTOM_TF, label='left', closed='left')
        .agg(agg)
        .dropna(subset=['close'])
        .iloc[:-1]          # drop last incomplete candle
    )

    print(f'{symbol_label} | custom {CUSTOM_TF} | {len(df_custom):,} bars')
    t_custom = (f'{symbol_label} · {CUSTOM_TF} (custom)  '
                f'({df_custom.index[0].date()} → {df_custom.index[-1].date()})')
    plot_candles(df_custom, t_custom, show_volume=True, show_vwap=True).show()

In [ ]:
# ── 9. Intraday volume heatmap ────────────────────────────────────────────────
# Shows average volume by hour-of-day across the last N days.
# Useful for identifying high-liquidity windows (UTC time).

HEATMAP_DAYS = 90   # days of 1m data to analyse

df_1m_heat = _load_raw(BASE)
if df_1m_heat is None or df_1m_heat.empty:
    print(f'⚠  No 1m raw data for {BASE}')
else:
    cutoff_h = df_1m_heat.index[-1] - pd.Timedelta(days=HEATMAP_DAYS)
    df_h     = df_1m_heat[df_1m_heat.index >= cutoff_h].copy()
    df_h['hour'] = df_h.index.hour

    avg_vol_by_hour = df_h.groupby('hour')['volume'].mean()

    fig_hm = go.Figure(go.Bar(
        x=avg_vol_by_hour.index,
        y=avg_vol_by_hour.values,
        marker_color=avg_vol_by_hour.values,
        marker_colorscale='Viridis',
        name='Avg Volume',
    ))
    fig_hm.update_layout(
        title=f'{symbol_label} · Avg volume by hour-of-day (UTC, last {HEATMAP_DAYS}d)',
        xaxis_title='Hour (UTC)', yaxis_title='Avg Volume',
        template='plotly_dark', height=350,
        xaxis=dict(tickmode='linear', tick0=0, dtick=1),
    )
    fig_hm.show()

In [ ]:
# ── 10. Multi-symbol snapshot ─────────────────────────────────────────────────
# Quick overview candle chart for all stored symbols at a single timeframe.

SNAP_TF   = '1h'    # timeframe to use for snapshot
SNAP_DAYS = 14      # lookback days per chart

for sym in CRYPTO_SYMBOLS:
    try:
        df_s = load_crypto_ohlcv(sym, SNAP_TF)
        if df_s is None or df_s.empty:
            print(f'  ⚠  {sym}: no data')
            continue

        cutoff_s = df_s.index[-1] - pd.Timedelta(days=SNAP_DAYS)
        df_s     = df_s[df_s.index >= cutoff_s]

        if df_s.empty:
            continue

        pct_chg = (df_s['close'].iloc[-1] / df_s['close'].iloc[0] - 1) * 100
        t_s = (f'{sym}USDT PERP · {SNAP_TF}  '
               f'({df_s.index[0].date()} → {df_s.index[-1].date()})  '
               f'Δ {pct_chg:+.2f}%')
        plot_candles(df_s, t_s, show_volume=True, show_vwap=False).show()

    except Exception as e:
        print(f'  ✗  {sym}: {e}')